In [ ]:
This notebook develops a clustering workflow using K-Means and hierarchical clustering on the `Real_Estate_Project.csv` dataset.</VSCode.Cell>
<VSCode.Cell language="python">import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 120})</VSCode.Cell>
<VSCode.Cell language="markdown">## Load and Preprocess Dataset
Load the data, inspect the feature names, and clean numeric columns for clustering.</VSCode.Cell>
<VSCode.Cell language="python">csv_path = "Real_Estate_Project.csv"
df = pd.read_csv(csv_path, engine="python")
df.columns = df.columns.str.strip()

df.head()
</VSCode.Cell>
<VSCode.Cell language="python">print("Shape:", df.shape)
print("Columns:", list(df.columns))
</VSCode.Cell>
<VSCode.Cell language="python"># Clean sale_price and numeric feature columns
if "sale_price" in df.columns:
    df["sale_price"] = (
        df["sale_price"].astype(str)
        .str.replace("\$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

numeric_features = [
    "floor_area_sqft",
    "sale_price",
    "Individual",
    "Company",
    "Home",
    "Investment",
    "Country Encoding",
    "Region Encoding",
    "Sacled Satsifaction Score",
    "satisfaction",
]

features = [col for col in numeric_features if col in df.columns]
for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

cluster_df = df[features].copy().dropna()
cluster_df.head()
</VSCode.Cell>
<VSCode.Cell language="python">print("Features used for clustering:", features)
print("Cleaned shape:", cluster_df.shape)
</VSCode.Cell>
<VSCode.Cell language="python">scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)
</VSCode.Cell>
<VSCode.Cell language="markdown">## Select Number of Clusters
Use the elbow method and a hierarchical dendrogram to select the cluster count.</VSCode.Cell>
<VSCode.Cell language="python">inertia = []
cluster_range = range(2, 8)
for k in cluster_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.plot(cluster_range, inertia, marker="o")
plt.title("K-Means Elbow Method")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.xticks(cluster_range)
plt.show()
</VSCode.Cell>
<VSCode.Cell language="python">Z = linkage(X_scaled, method="ward")
dendrogram(Z, truncate_mode="level", p=5)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Sample index or node")
plt.ylabel("Distance")
plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">## Perform K-means Clustering
Fit K-Means and inspect the cluster centers in the scaled feature space.</VSCode.Cell>
<VSCode.Cell language="python">best_k = 4  # Choose based on elbow and dendrogram results
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels_kmeans = kmeans.fit_predict(X_scaled)
cluster_df["kmeans_cluster"] = cluster_labels_kmeans

print("K-means cluster counts:")
print(pd.Series(cluster_labels_kmeans).value_counts().sort_index())
print("\nCluster centers (scaled):")
print(pd.DataFrame(kmeans.cluster_centers_, columns=features))
</VSCode.Cell>
<VSCode.Cell language="markdown">## Perform Hierarchical Clustering
Compute agglomerative clusters and compare labels against K-Means.</VSCode.Cell>
<VSCode.Cell language="python">hierarchical = AgglomerativeClustering(n_clusters=best_k, affinity="euclidean", linkage="ward")
cluster_labels_agglom = hierarchical.fit_predict(X_scaled)
cluster_df["hierarchical_cluster"] = cluster_labels_agglom

print("Hierarchical cluster counts:")
print(pd.Series(cluster_labels_agglom).value_counts().sort_index())
</VSCode.Cell>
<VSCode.Cell language="markdown">## Compare Clustering Results
Compare membership overlap and create a simple cross-tabulation table.</VSCode.Cell>
<VSCode.Cell language="python">comparison = pd.crosstab(cluster_df["kmeans_cluster"], cluster_df["hierarchical_cluster"], rownames=["KMeans"], colnames=["Hierarchical"])
comparison
</VSCode.Cell>
<VSCode.Cell language="markdown">## Visualize Cluster Assignments
Plot the cluster assignments and inspect any clear structure in the real estate features.</VSCode.Cell>
<VSCode.Cell language="python">plot_df = cluster_df.copy()
plot_df["kmeans_cluster"] = plot_df["kmeans_cluster"].astype(str)

sns.scatterplot(
    data=plot_df,
    x="floor_area_sqft",
    y="sale_price",
    hue="kmeans_cluster",
    palette="tab10",
    alpha=0.85,
    edgecolor="k",
)
plt.title("K-Means Clusters: Floor Area vs Sale Price")
plt.show()
</VSCode.Cell>
<VSCode.Cell language="python">sns.pairplot(
    plot_df.sample(n=min(300, len(plot_df))),
    vars=["floor_area_sqft", "sale_price", "Sacled Satsifaction Score", "satisfaction"],
    hue="kmeans_cluster",
    palette="tab10",
    diag_kind="kde",
)
plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">## Evaluate Cluster Quality
Compute silhouette scores and compare cluster quality across different values of k.</VSCode.Cell>
<VSCode.Cell language="python">silhouette_kmeans = silhouette_score(X_scaled, cluster_labels_kmeans)
print(f"K-Means silhouette score: {silhouette_kmeans:.4f}")

for k in cluster_range:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f"k={k}, silhouette={score:.4f}")
